# Campo eléctrico FIRI de referencia para distintos materiales — NAA–PLO

Este notebook reemplaza la versión antigua de `generate-reference-efield.ipynb`.

Objetivo único:

1. construir la misma geometría de **10 segmentos** usada por la inversión Whittaker del background;
2. evaluar FIRI e IGRF en los puntos medios correctos;
3. generar, desde las 00:00 UTC, una serie diaria de amplitud y fase en PLO para cada configuración del suelo;
4. desenvolver cada fase desde el inicio del día;
5. alinear todas las ramas con la referencia oceánica mediante un único múltiplo entero de \(2\pi\);
6. guardar un archivo JLD2 individual por configuración con las claves compatibles `amp`, `phi` y `zdt`.

La matriz del suelo se ordena siempre desde el transmisor NAA hacia el receptor PLO.  
El notebook no contiene funciones de inversión, ajustes polinómicos ni cálculo de perturbaciones.

In [1]:
################################################################################
# 1. Entorno y dependencias
################################################################################

include("./mytools/preamble.jl")
include("./mytools/scenario.jl")

using Distributed
using Dates
using TimeZones
using LinearAlgebra
using Statistics
using Printf
using DataFrames
using JLD2
using DSP
using Interpolations
import CairoMakie as CM

ENV["OPENBLAS_NUM_THREADS"] = "1"
ENV["MKL_NUM_THREADS"] = "1"
ENV["BLIS_NUM_THREADS"] = "1"
ENV["OMP_NUM_THREADS"] = "1"
LinearAlgebra.BLAS.set_num_threads(1)

println("Julia threads en Main: ", Threads.nthreads())
println("BLAS threads en Main:  ", LinearAlgebra.BLAS.get_num_threads())

[ Info: Loading CHAOS-7.11.mat


Geodesic Distance from WWVB to PLO: 6568.28 km
Geodesic Distance from WWVB to PIU: 5659.0 km
Geodesic Distance from NAA to PLO: 6568.28 km
Geodesic Distance from NAA to PIU: 5659.0 km
Initial time: 2008-01-10T10:00:00+00:00
Scenario defined.
Datetime range: from 2008-01-10 10:00:00 UTC to 2008-01-11 10:00:00 UTC with a length of 289.
Julia threads en Main: 8
BLAS threads en Main:  1


In [ ]:
################################################################################
# 2. Configuración del experimento y casos de suelo
################################################################################

N_WAVEGUIDES_TARGET = 10
GEOLINSPACE_NPOINTS_ARGUMENT = 19
FIRI_FINDEX = 75
CADENCE = Minute(5)
INCLUDE_NEXT_MIDNIGHT = true

DATE_UTC = Date(2008, 3, 25)

OUTPUT_DIR = "outputs_reference_efield_firi_10wg_materials_20080325"
CHECKPOINT_BATCH_SIZE = 100
RESUME_IF_AVAILABLE = true
FRESH_RUN = false
WRITE_LEGACY_OCEAN_ALIAS = false

mkpath(OUTPUT_DIR)

ground_indices_catalog = collect(keys(GROUND))
ground_indices_catalog == collect(1:10) || error(
    "Se esperaban exactamente los materiales GROUND[1:10]."
)

expected_epsr = [5.0, 5.0, 10.0, 10.0, 15.0, 15.0, 15.0, 15.0, 15.0, 81.0]
expected_sigma = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 4.0]

actual_epsr = Float64[GROUND[i].ϵᵣ for i in 1:10]
actual_sigma = Float64[GROUND[i].σ for i in 1:10]

all(isapprox.(actual_epsr, expected_epsr; atol=0.0, rtol=0.0)) || error(
    "Las permitividades de GROUND no coinciden con el catálogo esperado."
)
all(isapprox.(actual_sigma, expected_sigma; atol=0.0, rtol=1e-12)) || error(
    "Las conductividades de GROUND no coinciden con el catálogo esperado."
)

ground_names = Dict(
    1 => "Ice / permafrost",
    3 => "Poor soil / fresh water",
    5 => "Average soil",
    7 => "Clay / good soil",
    9 => "Marine sands",
    10 => "Sea water",
)

df_ground_catalog = DataFrame(
    ground_index = collect(1:10),
    epsilon_r = actual_epsr,
    sigma_S_m = actual_sigma,
    material = [get(ground_names, i, "Intermediate LMP/LWPC class") for i in 1:10],
)

case_keys = String[]
case_labels = String[]
case_types = String[]
case_ground_indices = Vector{Vector{Int}}()

function add_ground_case!(
    case_keys,
    case_labels,
    case_types,
    case_ground_indices,
    key::AbstractString,
    label::AbstractString,
    case_type::AbstractString,
    indices::AbstractVector{<:Integer},
    n_segments::Integer,
)
    length(indices) == n_segments || error(
        "El caso '$key' contiene $(length(indices)) materiales; se requieren $n_segments."
    )
    all(i -> i in 1:10, indices) || error(
        "El caso '$key' contiene un índice GROUND inválido."
    )
    key in case_keys && error("El identificador '$key' está repetido.")

    push!(case_keys, String(key))
    push!(case_labels, String(label))
    push!(case_types, String(case_type))
    push!(case_ground_indices, Int.(indices))
    return nothing
end

# Diez materiales homogéneos.
for i in 1:10
    add_ground_case!(
        case_keys,
        case_labels,
        case_types,
        case_ground_indices,
        "ground_$(i)",
        "GROUND[$i]: $(get(ground_names, i, "intermediate class"))",
        "homogeneous",
        fill(i, N_WAVEGUIDES_TARGET),
        N_WAVEGUIDES_TARGET,
    )
end

# Casos heterogéneos sobre los mismos diez segmentos de la inversión Whittaker.
add_ground_case!(
    case_keys, case_labels, case_types, case_ground_indices,
    "ocean5_goodsoil5",
    "G10 × 5 → G7 × 5",
    "heterogeneous",
    vcat(fill(10, 5), fill(7, 5)),
    N_WAVEGUIDES_TARGET,
)
add_ground_case!(
    case_keys, case_labels, case_types, case_ground_indices,
    "goodsoil5_ocean5",
    "G7 × 5 → G10 × 5",
    "heterogeneous",
    vcat(fill(7, 5), fill(10, 5)),
    N_WAVEGUIDES_TARGET,
)
add_ground_case!(
    case_keys, case_labels, case_types, case_ground_indices,
    "ocean7_goodsoil3",
    "G10 × 7 → G7 × 3",
    "heterogeneous",
    vcat(fill(10, 7), fill(7, 3)),
    N_WAVEGUIDES_TARGET,
)
add_ground_case!(
    case_keys, case_labels, case_types, case_ground_indices,
    "goodsoil3_ocean7",
    "G7 × 3 → G10 × 7",
    "heterogeneous",
    vcat(fill(7, 3), fill(10, 7)),
    N_WAVEGUIDES_TARGET,
)
add_ground_case!(
    case_keys, case_labels, case_types, case_ground_indices,
    "ocean6_goodsoil4",
    "G10 × 6 → G7 × 4",
    "heterogeneous",
    vcat(fill(10, 6), fill(7, 4)),
    N_WAVEGUIDES_TARGET,
)
add_ground_case!(
    case_keys, case_labels, case_types, case_ground_indices,
    "goodsoil4_ocean6",
    "G7 × 4 → G10 × 6",
    "heterogeneous",
    vcat(fill(7, 4), fill(10, 6)),
    N_WAVEGUIDES_TARGET,
)

# Puede reducirse esta lista sin modificar el resto del notebook.
RUN_CASE_KEYS = copy(case_keys)

selected_positions = [findfirst(==(key), case_keys) for key in RUN_CASE_KEYS]
any(isnothing, selected_positions) && error("RUN_CASE_KEYS contiene un caso desconocido.")
selected_positions = Int.(selected_positions)

case_keys = case_keys[selected_positions]
case_labels = case_labels[selected_positions]
case_types = case_types[selected_positions]
case_ground_indices = case_ground_indices[selected_positions]

"ground_10" in case_keys || error(
    "Debe incluirse ground_10 porque define la rama oceánica común."
)

N_CASES = length(case_keys)

case_ground_matrix = Matrix{Int}(undef, N_CASES, N_WAVEGUIDES_TARGET)
for i in 1:N_CASES
    case_ground_matrix[i, :] .= case_ground_indices[i]
end

df_cases = DataFrame(
    case_index = collect(1:N_CASES),
    case_key = case_keys,
    case_label = case_labels,
    case_type = case_types,
    ground_indices_TX_to_RX = [join(v, ",") for v in case_ground_indices],
)

display(df_ground_catalog)
display(df_cases)

In [ ]:
################################################################################
# 3. Geometría NAA–PLO y grilla temporal diaria
################################################################################

lats_path, lons_path, dists_path = geolinspace(
    "NAA",
    "PLO",
    GEOLINSPACE_NPOINTS_ARGUMENT,
    false,
)

N_PATH_POINTS_EFFECTIVE = length(dists_path)
N_PATH_POINTS_EFFECTIVE == GEOLINSPACE_NPOINTS_ARGUMENT + 2 || error(
    "geolinspace devolvió un número inesperado de puntos."
)

lats_mid = Float64.(lats_path[2:2:end])
lons_mid = Float64.(lons_path[2:2:end])
dists_mid = Float64.(dists_path[2:2:end])

Nwaveguide = length(dists_mid)
Nwaveguide == N_WAVEGUIDES_TARGET || error(
    "La geometría contiene $Nwaveguide segmentos; se requieren $N_WAVEGUIDES_TARGET."
)

dists_start_all = Float64.(dists_path[1:2:end])
dists_start = dists_start_all[1:Nwaveguide]

dists_end = Float64.(dists_path[3:2:end])
segment_lengths_km = (dists_end .- dists_start) ./ 1e3

length(lats_mid) == Nwaveguide || error("Latitudes medias incompatibles.")
length(lons_mid) == Nwaveguide || error("Longitudes medias incompatibles.")
length(dists_start) == Nwaveguide || error("Inicios de segmento incompatibles.")
length(dists_end) == Nwaveguide || error("Finales de segmento incompatibles.")
first(dists_start) == 0.0 || error("El primer segmento no comienza en cero.")
issorted(dists_start) || error("Los inicios de segmento no están ordenados.")
all(segment_lengths_km .> 0.0) || error("Existe un segmento de longitud no positiva.")

isdefined(Main, :rx12) || error(
    "scenario.jl no define rx12, el GroundSampler correspondiente a NAA–PLO."
)
rx_naa_plo = rx12

lon_rx = rx_params["Longitude"]
lat_rx = rx_params["Latitude"]
rx_rec = Receiver("PLO", lat_rx, lon_rx, 0.0, VerticalDipole())

path_length_m = Float64(last(dists_path))

if isdefined(Main, :d2_radar)
    if !isapprox(Float64(d2_radar), path_length_m; atol=2e3, rtol=5e-4)
        @warn(
            "d2_radar y el extremo de geolinspace difieren",
            d2_radar = Float64(d2_radar),
            geolinspace_path_length_m = path_length_m,
        )
    end
end

zdt_start = ZonedDateTime(
    year(DATE_UTC),
    month(DATE_UTC),
    day(DATE_UTC),
    0,
    0,
    0,
    tz"UTC",
)

zdt_stop = INCLUDE_NEXT_MIDNIGHT ?
    zdt_start + Day(1) :
    zdt_start + Day(1) - CADENCE

zdt_eval = collect(datelinspace(zdt_start, zdt_stop, CADENCE))
Nt = length(zdt_eval)

expected_Nt = INCLUDE_NEXT_MIDNIGHT ? 289 : 288
Nt == expected_Nt || error(
    "La grilla temporal contiene $Nt puntos; se esperaban $expected_Nt."
)
issorted(zdt_eval) || error("La grilla temporal no está ordenada.")
allunique(zdt_eval) || error("La grilla temporal contiene valores repetidos.")

scenario_summary = DataFrame(
    transmitter = [tx_params2["Prefix"]],
    receiver = [rx_params["Prefix"]],
    path_length_km = [path_length_m / 1e3],
    n_waveguides = [Nwaveguide],
    geolinspace_npoints_argument = [GEOLINSPACE_NPOINTS_ARGUMENT],
    effective_path_points = [N_PATH_POINTS_EFFECTIVE],
    first_UTC = [first(zdt_eval)],
    last_UTC = [last(zdt_eval)],
    cadence_min = [Dates.value(CADENCE)],
    n_times = [Nt],
    firi_findex = [FIRI_FINDEX],
    receiver_sampler = ["rx12 (NAA–PLO)"],
)

display(scenario_summary)
println("Longitudes de los segmentos [km]: ", round.(segment_lengths_km; digits=3))

In [ ]:
################################################################################
# 4. Precomputaciones comunes: IGRF una sola vez y FIRI en puntos medios
################################################################################

println("Calculando IGRF en los $Nwaveguide puntos medios...")

# La llamada usada por el proyecto depende del año, no de cada minuto del día.
bf_mid = igrf(
    tx2,
    rx_rec,
    year(first(zdt_eval)),
    dists_mid;
    alt = 60e3,
)

length(bf_mid) == Nwaveguide || error(
    "IGRF devolvió $(length(bf_mid)) valores; se esperaban $Nwaveguide."
)

println("Calculando FIRI para $Nt tiempos y $Nwaveguide segmentos...")

nefiri_mid = Array{Float64}(
    undef,
    length(alt),
    Nwaveguide,
    Nt,
)

firi_elapsed_s = @elapsed begin
    for (k, tt) in enumerate(zdt_eval)
        ne_temp = Float64.(nefiri(
            lat = lats_mid,
            lon = lons_mid,
            zdt = tt,
            findex = FIRI_FINDEX,
        ))

        size(ne_temp) == (length(alt), Nwaveguide) || error(
            "Dimensiones FIRI inesperadas en $tt: $(size(ne_temp))."
        )
        all(isfinite, ne_temp) || error("FIRI produjo NaN/Inf en $tt.")
        all(ne_temp .>= 0.0) || error("FIRI produjo densidades negativas en $tt.")

        @views nefiri_mid[:, :, k] .= ne_temp
    end
end

println("FIRI precalculado en ", round(firi_elapsed_s; digits=2), " s.")
println(
    "Tamaño aproximado de nefiri_mid: ",
    round(Base.summarysize(nefiri_mid) / 2.0^20; digits=2),
    " MiB.",
)

In [ ]:
################################################################################
# 5. Workers de un hilo y contexto distribuido
################################################################################

allocated_cpus = parse(
    Int,
    get(ENV, "SLURM_CPUS_PER_TASK", string(Sys.CPU_THREADS)),
)

requested_workers = max(allocated_cpus - 1, 1)
N_WORKERS = min(requested_workers, N_CASES * Nt)

existing_workers = filter(!=(1), workers())
if !isempty(existing_workers)
    println("Eliminando workers existentes: ", existing_workers)
    rmprocs(existing_workers)
end

println(
    "Iniciando $N_WORKERS workers de un hilo ",
    "(CPU disponibles detectadas: $allocated_cpus)..."
)

addprocs(
    N_WORKERS;
    exeflags = "--threads=1",
    enable_threaded_blas = false,
)

eval_workers = workers()
length(eval_workers) == N_WORKERS || error(
    "Se esperaban $N_WORKERS workers; se encontraron $(length(eval_workers))."
)

workdir_path = abspath(pwd())

@everywhere begin
    using Distributed
    using LinearAlgebra
    using Interpolations
    using LongwaveModePropagator

    ENV["OPENBLAS_NUM_THREADS"] = "1"
    ENV["MKL_NUM_THREADS"] = "1"
    ENV["BLIS_NUM_THREADS"] = "1"
    ENV["OMP_NUM_THREADS"] = "1"

    LinearAlgebra.BLAS.set_num_threads(1)

    if myid() != 1
        cd($workdir_path)
    end
end

worker_configuration = [
    remotecall_fetch(pid) do
        (
            worker_id = myid(),
            julia_threads = Threads.nthreads(),
            blas_threads = LinearAlgebra.BLAS.get_num_threads(),
            lmp_loaded = isdefined(Main, :LongwaveModePropagator),
            ground_count = length(LongwaveModePropagator.GROUND),
            working_directory = pwd(),
        )
    end
    for pid in eval_workers
]

all(x.julia_threads == 1 for x in worker_configuration) || error(
    "Al menos un worker tiene más de un hilo Julia."
)
all(x.blas_threads == 1 for x in worker_configuration) || error(
    "Al menos un worker tiene BLAS multihilo."
)
all(x.lmp_loaded for x in worker_configuration) || error(
    "LongwaveModePropagator no está cargado en todos los workers."
)
all(x.ground_count == 10 for x in worker_configuration) || error(
    "El catálogo GROUND no coincide entre los workers."
)

display(DataFrame(worker_configuration))

In [ ]:
################################################################################
# 6. Evaluación de una combinación (caso de suelo, tiempo)
################################################################################

@everywhere function install_firi_reference_context!(
    alt_grid,
    nefiri_mid,
    bf_mid,
    dists_start,
    tx,
    rx,
    ground_matrix,
)
    global FIRIREF_ALT_GRID = alt_grid
    global FIRIREF_NEFIRI_MID = nefiri_mid
    global FIRIREF_BF_MID = bf_mid
    global FIRIREF_DISTS_START = dists_start
    global FIRIREF_TX = tx
    global FIRIREF_RX = rx
    global FIRIREF_GROUND_MATRIX = ground_matrix

    return (
        worker_id = myid(),
        n_altitudes = length(FIRIREF_ALT_GRID),
        n_segments = size(FIRIREF_NEFIRI_MID, 2),
        n_times = size(FIRIREF_NEFIRI_MID, 3),
        n_cases = size(FIRIREF_GROUND_MATRIX, 1),
    )
end

@everywhere function evaluate_firi_reference_job(
    case_index::Int,
    time_index::Int,
)
    t_start = time()

    alt_grid = FIRIREF_ALT_GRID
    nefiri_mid = FIRIREF_NEFIRI_MID
    bf_mid = FIRIREF_BF_MID
    dists_start = FIRIREF_DISTS_START
    tx = FIRIREF_TX
    rx = FIRIREF_RX

    ground_indices = collect(@view FIRIREF_GROUND_MATRIX[case_index, :])

    n_segments = length(dists_start)

    length(ground_indices) == n_segments || error(
        "El número de materiales no coincide con el número de segmentos."
    )
    length(bf_mid) == n_segments || error(
        "El número de campos IGRF no coincide con los segmentos."
    )
    size(nefiri_mid, 1) == length(alt_grid) || error(
        "La dimensión vertical de FIRI no coincide con alt_grid."
    )

    species = [
        begin
            profile = collect(@view nefiri_mid[:, j, time_index])

            all(isfinite, profile) || error(
                "Perfil FIRI no finito: segmento=$j, tiempo=$time_index."
            )
            minimum(profile) >= 0.0 || error(
                "Perfil FIRI negativo: segmento=$j, tiempo=$time_index."
            )

            ne_interp = Interpolations.interpolate(
                alt_grid,
                profile,
                FritschButlandMonotonicInterpolation(),
            )

            LongwaveModePropagator.Species(
                LongwaveModePropagator.QE,
                LongwaveModePropagator.ME,
                ne_interp,
                LongwaveModePropagator.electroncollisionfrequency,
            )
        end
        for j in 1:n_segments
    ]

    waveguide = LongwaveModePropagator.SegmentedWaveguide([
        LongwaveModePropagator.HomogeneousWaveguide(
            bf_mid[j],
            species[j],
            LongwaveModePropagator.GROUND[ground_indices[j]],
            dists_start[j],
        )
        for j in 1:n_segments
    ])

    _, amplitude_dB, phase_wrapped_rad =
        LongwaveModePropagator.propagate(waveguide, tx, rx)

    return (
        amplitude_dB = Float64(amplitude_dB),
        phase_wrapped_rad = Float64(phase_wrapped_rad),
        runtime_s = time() - t_start,
    )
end

@everywhere function evaluate_firi_reference_job_safe(job)
    case_index, time_index = job

    try
        result = evaluate_firi_reference_job(case_index, time_index)

        return (
            ok = true,
            case_index = case_index,
            time_index = time_index,
            amplitude_dB = result.amplitude_dB,
            phase_wrapped_rad = result.phase_wrapped_rad,
            runtime_s = result.runtime_s,
            worker_id = myid(),
            error_message = "",
        )
    catch err
        bt = catch_backtrace()

        return (
            ok = false,
            case_index = case_index,
            time_index = time_index,
            amplitude_dB = NaN,
            phase_wrapped_rad = NaN,
            runtime_s = NaN,
            worker_id = myid(),
            error_message = sprint(showerror, err, bt),
        )
    end
end

context_status = [
    remotecall_fetch(
        install_firi_reference_context!,
        pid,
        collect(alt),
        nefiri_mid,
        bf_mid,
        dists_start,
        tx2,
        rx_naa_plo,
        case_ground_matrix,
    )
    for pid in eval_workers
]

all(x.n_segments == Nwaveguide for x in context_status) || error(
    "El contexto de algún worker tiene un número incorrecto de segmentos."
)
all(x.n_times == Nt for x in context_status) || error(
    "El contexto de algún worker tiene un número incorrecto de tiempos."
)
all(x.n_cases == N_CASES for x in context_status) || error(
    "El contexto de algún worker tiene un número incorrecto de casos."
)

pilot_result = remotecall_fetch(
    evaluate_firi_reference_job_safe,
    first(eval_workers),
    (1, 1),
)

pilot_result.ok || error(
    "Falló la propagación piloto:\n$(pilot_result.error_message)"
)
isfinite(pilot_result.amplitude_dB) || error("Amplitud piloto no finita.")
isfinite(pilot_result.phase_wrapped_rad) || error("Fase piloto no finita.")

println(
    "Propagación piloto correcta: A=",
    round(pilot_result.amplitude_dB; digits=4),
    " dB, ϕ=",
    round(rad2deg(pilot_result.phase_wrapped_rad); digits=4),
    " deg.",
)

In [ ]:
################################################################################
# 7. Corrida distribuida reiniciable y checkpoint
################################################################################

checkpoint_file = joinpath(
    OUTPUT_DIR,
    "checkpoint_reference_efield_firi_10wg_20080325.jld2",
)

amplitude_matrix = fill(NaN, N_CASES, Nt)
phase_wrapped_raw_matrix = fill(NaN, N_CASES, Nt)
runtime_matrix_s = fill(NaN, N_CASES, Nt)
worker_matrix = fill(0, N_CASES, Nt)
completed_matrix = falses(N_CASES, Nt)

function save_reference_checkpoint!(
    filename,
    amplitude_matrix,
    phase_wrapped_raw_matrix,
    runtime_matrix_s,
    worker_matrix,
    completed_matrix,
    case_keys,
    case_ground_matrix,
    zdt_eval,
)
    tmp = filename * ".tmp"

    JLD2.jldsave(
        tmp;
        amplitude_matrix = amplitude_matrix,
        phase_wrapped_raw_matrix = phase_wrapped_raw_matrix,
        runtime_matrix_s = runtime_matrix_s,
        worker_matrix = worker_matrix,
        completed_matrix = completed_matrix,
        case_keys = case_keys,
        case_ground_matrix = case_ground_matrix,
        zdt_eval = zdt_eval,
    )

    mv(tmp, filename; force=true)
    return filename
end

if FRESH_RUN && isfile(checkpoint_file)
    rm(checkpoint_file)
end

if RESUME_IF_AVAILABLE && isfile(checkpoint_file)
    println("Cargando checkpoint existente: $checkpoint_file")

    checkpoint_data = JLD2.load(checkpoint_file)

    checkpoint_data["case_keys"] == case_keys || error(
        "El checkpoint corresponde a otros casos."
    )
    checkpoint_data["case_ground_matrix"] == case_ground_matrix || error(
        "El checkpoint corresponde a otra matriz de materiales."
    )
    checkpoint_data["zdt_eval"] == zdt_eval || error(
        "El checkpoint corresponde a otra grilla temporal."
    )

    amplitude_matrix .= checkpoint_data["amplitude_matrix"]
    phase_wrapped_raw_matrix .= checkpoint_data["phase_wrapped_raw_matrix"]
    runtime_matrix_s .= checkpoint_data["runtime_matrix_s"]
    worker_matrix .= checkpoint_data["worker_matrix"]
    completed_matrix .= checkpoint_data["completed_matrix"]
end

function run_reference_batches!(
    pool,
    pending_jobs,
    amplitude_matrix,
    phase_wrapped_raw_matrix,
    runtime_matrix_s,
    worker_matrix,
    completed_matrix,
    checkpoint_file,
    batch_size,
    case_keys,
    case_ground_matrix,
    zdt_eval,
)
    isempty(pending_jobs) && return nothing

    n_total = length(pending_jobs)
    n_received = 0
    wall_start = time()

    for first_index in 1:batch_size:n_total
        last_index = min(first_index + batch_size - 1, n_total)
        batch = pending_jobs[first_index:last_index]

        batch_results = pmap(
            evaluate_firi_reference_job_safe,
            pool,
            batch;
            batch_size = 1,
        )

        failures = filter(r -> !r.ok, batch_results)

        for result in batch_results
            if result.ok
                c = result.case_index
                t = result.time_index

                amplitude_matrix[c, t] = result.amplitude_dB
                phase_wrapped_raw_matrix[c, t] = result.phase_wrapped_rad
                runtime_matrix_s[c, t] = result.runtime_s
                worker_matrix[c, t] = result.worker_id
                completed_matrix[c, t] = true
            end
        end

        save_reference_checkpoint!(
            checkpoint_file,
            amplitude_matrix,
            phase_wrapped_raw_matrix,
            runtime_matrix_s,
            worker_matrix,
            completed_matrix,
            case_keys,
            case_ground_matrix,
            zdt_eval,
        )

        n_received += length(batch_results)
        elapsed_min = (time() - wall_start) / 60
        rate = n_received / max(time() - wall_start, eps(Float64))
        remaining = n_total - n_received
        eta_min = remaining / max(rate, eps(Float64)) / 60

        @printf(
            "Progress: %5d/%5d | elapsed=%.1f min | ETA≈%.1f min\n",
            n_received,
            n_total,
            elapsed_min,
            eta_min,
        )

        if !isempty(failures)
            details = join(
                [
                    "case=$(r.case_index), time=$(r.time_index), worker=$(r.worker_id)\n$(r.error_message)"
                    for r in failures
                ],
                "\n\n",
            )
            error("Fallaron $(length(failures)) propagaciones:\n$details")
        end
    end

    return nothing
end

pending_jobs = [
    (case_index, time_index)
    for case_index in 1:N_CASES
    for time_index in 1:Nt
    if !completed_matrix[case_index, time_index]
]

println(
    "Trabajos pendientes: ",
    length(pending_jobs),
    " / ",
    N_CASES * Nt,
)

pool = CachingPool(eval_workers)

run_reference_batches!(
    pool,
    pending_jobs,
    amplitude_matrix,
    phase_wrapped_raw_matrix,
    runtime_matrix_s,
    worker_matrix,
    completed_matrix,
    checkpoint_file,
    CHECKPOINT_BATCH_SIZE,
    case_keys,
    case_ground_matrix,
    zdt_eval,
)

all(completed_matrix) || error("No se completaron todas las propagaciones.")
all(isfinite, amplitude_matrix) || error("La matriz de amplitud contiene NaN/Inf.")
all(isfinite, phase_wrapped_raw_matrix) || error("La matriz de fase contiene NaN/Inf.")

println("Propagaciones completadas: ", count(completed_matrix))

In [ ]:
################################################################################
# 8. Unwrapping temporal y alineamiento de rama con el océano a las 00:00 UTC
################################################################################

function build_phase_references(
    phase_wrapped_raw_matrix,
    case_keys,
    ocean_key::AbstractString,
)
    n_cases, n_times = size(phase_wrapped_raw_matrix)

    ocean_index = findfirst(==(ocean_key), case_keys)
    ocean_index === nothing && error("No existe el caso oceánico '$ocean_key'.")

    phase_unwrapped_raw_matrix = Matrix{Float64}(undef, n_cases, n_times)

    for i in 1:n_cases
        phase_unwrapped_raw_matrix[i, :] .= DSP.unwrap(
            collect(@view phase_wrapped_raw_matrix[i, :]);
            dims = 1,
            discont = 0.9π,
            period = 2π,
        )
    end

    ocean_unwrapped = @view phase_unwrapped_raw_matrix[ocean_index, :]

    branch_offset_cycles = Vector{Int}(undef, n_cases)
    phase_reference_rad_matrix = Matrix{Float64}(undef, n_cases, n_times)
    phase_unwrapped_aligned_matrix = Matrix{Float64}(undef, n_cases, n_times)

    for i in 1:n_cases
        # Un único entero para toda la serie. La rama se fija en el primer punto,
        # 00:00 UTC, y no se reajusta independientemente en cada tiempo.
        branch_offset_cycles[i] = round(
            Int,
            (ocean_unwrapped[1] - phase_unwrapped_raw_matrix[i, 1]) / (2π),
        )

        phase_reference_rad_matrix[i, :] .=
            phase_wrapped_raw_matrix[i, :] .+
            branch_offset_cycles[i] * 2π

        phase_unwrapped_aligned_matrix[i, :] .= DSP.unwrap(
            collect(@view phase_reference_rad_matrix[i, :]);
            dims = 1,
            discont = 0.9π,
            period = 2π,
        )

        expected = phase_unwrapped_raw_matrix[i, :] .+
            branch_offset_cycles[i] * 2π

        maximum(abs.(
            phase_unwrapped_aligned_matrix[i, :] .- expected
        )) <= 1e-10 || error(
            "El alineamiento de fase falló para $(case_keys[i])."
        )
    end

    return (
        ocean_index = ocean_index,
        phase_unwrapped_raw_matrix = phase_unwrapped_raw_matrix,
        phase_reference_rad_matrix = phase_reference_rad_matrix,
        phase_unwrapped_aligned_matrix = phase_unwrapped_aligned_matrix,
        branch_offset_cycles = branch_offset_cycles,
    )
end

phase_results = build_phase_references(
    phase_wrapped_raw_matrix,
    case_keys,
    "ground_10",
)

ocean_index = phase_results.ocean_index
phase_unwrapped_raw_matrix = phase_results.phase_unwrapped_raw_matrix
phase_reference_rad_matrix = phase_results.phase_reference_rad_matrix
phase_unwrapped_aligned_matrix = phase_results.phase_unwrapped_aligned_matrix
branch_offset_cycles = phase_results.branch_offset_cycles
phase_unwrapped_aligned_deg_matrix = rad2deg.(phase_unwrapped_aligned_matrix)

ocean_amplitude = @view amplitude_matrix[ocean_index, :]
ocean_phase_deg = @view phase_unwrapped_aligned_deg_matrix[ocean_index, :]

df_phase_branch_audit = DataFrame(
    case_key = case_keys,
    case_label = case_labels,
    branch_offset_cycles = branch_offset_cycles,
    phase_difference_at_0000_deg = [
        phase_unwrapped_aligned_deg_matrix[i, 1] - ocean_phase_deg[1]
        for i in 1:N_CASES
    ],
    max_abs_phase_difference_vs_ocean_deg = [
        maximum(abs.(
            phase_unwrapped_aligned_deg_matrix[i, :] .- ocean_phase_deg
        ))
        for i in 1:N_CASES
    ],
)

display(df_phase_branch_audit)

In [ ]:
################################################################################
# 9. Figuras de control
################################################################################

t0 = DateTime(first(zdt_eval))
t_hours = Float64.(
    Dates.value.(DateTime.(zdt_eval) .- t0)
) ./ 3_600_000

xtick_hours = collect(0.0:2.0:24.0)
xtick_labels = [
    Dates.format(t0 + Minute(round(Int, 60h)), "HH:MM")
    for h in xtick_hours
]

homogeneous_indices = findall(==("homogeneous"), case_types)
heterogeneous_indices = findall(==("heterogeneous"), case_types)

fig_homogeneous = CM.Figure(size=(1550, 950))

ax_hom_A = CM.Axis(
    fig_homogeneous[1, 1],
    title = "Referencias FIRI homogéneas — amplitud",
    ylabel = "Amplitud [dB]",
    xticks = (xtick_hours, xtick_labels),
)

for i in homogeneous_indices
    CM.lines!(
        ax_hom_A,
        t_hours,
        amplitude_matrix[i, :],
        label = case_keys[i],
        linewidth = 2,
    )
end

CM.axislegend(ax_hom_A, position=:rt, nbanks=2)

ax_hom_phi = CM.Axis(
    fig_homogeneous[2, 1],
    title = "Referencias FIRI homogéneas — fase continua",
    xlabel = "Hora UTC",
    ylabel = "Fase [deg]",
    xticks = (xtick_hours, xtick_labels),
)

for i in homogeneous_indices
    CM.lines!(
        ax_hom_phi,
        t_hours,
        phase_unwrapped_aligned_deg_matrix[i, :],
        label = case_keys[i],
        linewidth = 2,
    )
end

CM.linkxaxes!(ax_hom_A, ax_hom_phi)

hom_png = joinpath(OUTPUT_DIR, "reference_firi_homogeneous_10wg_20080325.png")
hom_pdf = joinpath(OUTPUT_DIR, "reference_firi_homogeneous_10wg_20080325.pdf")
CM.save(hom_png, fig_homogeneous)
CM.save(hom_pdf, fig_homogeneous)

fig_mixed = CM.Figure(size=(1550, 950))

comparison_indices = unique(vcat(
    findall(key -> key in ("ground_10", "ground_7"), case_keys),
    heterogeneous_indices,
))

ax_mix_A = CM.Axis(
    fig_mixed[1, 1],
    title = "Referencias FIRI heterogéneas y controles — amplitud",
    ylabel = "Amplitud [dB]",
    xticks = (xtick_hours, xtick_labels),
)

for i in comparison_indices
    CM.lines!(
        ax_mix_A,
        t_hours,
        amplitude_matrix[i, :],
        label = case_keys[i],
        linewidth = 2,
    )
end

CM.axislegend(ax_mix_A, position=:rt, nbanks=2)

ax_mix_phi = CM.Axis(
    fig_mixed[2, 1],
    title = "Referencias FIRI heterogéneas y controles — fase continua",
    xlabel = "Hora UTC",
    ylabel = "Fase [deg]",
    xticks = (xtick_hours, xtick_labels),
)

for i in comparison_indices
    CM.lines!(
        ax_mix_phi,
        t_hours,
        phase_unwrapped_aligned_deg_matrix[i, :],
        label = case_keys[i],
        linewidth = 2,
    )
end

CM.linkxaxes!(ax_mix_A, ax_mix_phi)

mix_png = joinpath(OUTPUT_DIR, "reference_firi_heterogeneous_10wg_20080325.png")
mix_pdf = joinpath(OUTPUT_DIR, "reference_firi_heterogeneous_10wg_20080325.pdf")
CM.save(mix_png, fig_mixed)
CM.save(mix_pdf, fig_mixed)

display(fig_homogeneous)
display(fig_mixed)

In [ ]:
################################################################################
# 10. Guardado de referencias individuales y archivo consolidado
################################################################################

date_tag = Dates.format(DATE_UTC, "yyyymmdd")
individual_files = String[]

for i in 1:N_CASES
    key = case_keys[i]
    ground_indices_case = collect(@view case_ground_matrix[i, :])
    epsilon_r_by_segment = Float64[GROUND[j].ϵᵣ for j in ground_indices_case]
    sigma_S_m_by_segment = Float64[GROUND[j].σ for j in ground_indices_case]

    output_file = joinpath(
        OUTPUT_DIR,
        "reference-efield-firi_$(key)_10wg_$(date_tag).jld2",
    )

    # Compatibilidad con el script antiguo:
    #   amp -> amplitud [dB]
    #   phi -> fase de referencia en radianes, con rama global ya alineada
    #   zdt -> tiempos UTC
    JLD2.jldsave(
        output_file;
        amp = collect(@view amplitude_matrix[i, :]),
        phi = collect(@view phase_reference_rad_matrix[i, :]),
        zdt = zdt_eval,
        phi_wrapped_raw_rad = collect(@view phase_wrapped_raw_matrix[i, :]),
        phi_unwrapped_raw_rad = collect(@view phase_unwrapped_raw_matrix[i, :]),
        phi_unwrapped_aligned_rad = collect(
            @view phase_unwrapped_aligned_matrix[i, :]
        ),
        phi_unwrapped_aligned_deg = collect(
            @view phase_unwrapped_aligned_deg_matrix[i, :]
        ),
        phase_branch_offset_cycles = branch_offset_cycles[i],
        phase_branch_anchor_UTC = first(zdt_eval),
        phase_branch_anchor_case = "ground_10",
        case_key = key,
        case_label = case_labels[i],
        case_type = case_types[i],
        ground_indices_TX_to_RX = ground_indices_case,
        epsilon_r_by_segment = epsilon_r_by_segment,
        sigma_S_m_by_segment = sigma_S_m_by_segment,
        lats_mid = lats_mid,
        lons_mid = lons_mid,
        dists_mid_m = dists_mid,
        dists_start_m = dists_start,
        dists_end_m = dists_end,
        segment_lengths_km = segment_lengths_km,
        altitude_m = collect(alt),
        transmitter = tx_params2["Prefix"],
        receiver = rx_params["Prefix"],
        receiver_sampler = "rx12",
        path_length_m = path_length_m,
        n_waveguides = Nwaveguide,
        geolinspace_npoints_argument = GEOLINSPACE_NPOINTS_ARGUMENT,
        firi_findex = FIRI_FINDEX,
        cadence_minutes = Dates.value(CADENCE),
        include_next_midnight = INCLUDE_NEXT_MIDNIGHT,
        runtime_s_by_time = collect(@view runtime_matrix_s[i, :]),
        worker_id_by_time = collect(@view worker_matrix[i, :]),
    )

    # Verificación inmediata de las tres claves que consume la inversión.
    amp_check, phi_check, zdt_check = JLD2.jldopen(output_file, "r") do f
        for required_key in ("amp", "phi", "zdt")
            haskey(f, required_key) || error(
                "Falta la clave '$required_key' en $output_file."
            )
        end

        Float64.(vec(f["amp"])),
        Float64.(vec(f["phi"])),
        vec(f["zdt"])
    end

    amp_check == collect(@view amplitude_matrix[i, :]) || error(
        "La verificación de amplitud falló para $key."
    )
    phi_check == collect(@view phase_reference_rad_matrix[i, :]) || error(
        "La verificación de fase falló para $key."
    )
    zdt_check == zdt_eval || error(
        "La verificación temporal falló para $key."
    )

    push!(individual_files, output_file)
end

df_summary = DataFrame(
    case_index = collect(1:N_CASES),
    case_key = case_keys,
    case_label = case_labels,
    case_type = case_types,
    ground_indices_TX_to_RX = [join(v, ",") for v in case_ground_indices],
    branch_offset_cycles = branch_offset_cycles,
    mean_delta_A_vs_ocean_dB = [
        mean(amplitude_matrix[i, :] .- ocean_amplitude)
        for i in 1:N_CASES
    ],
    rmse_delta_A_vs_ocean_dB = [
        sqrt(mean((amplitude_matrix[i, :] .- ocean_amplitude).^2))
        for i in 1:N_CASES
    ],
    mean_delta_phi_vs_ocean_deg = [
        mean(
            phase_unwrapped_aligned_deg_matrix[i, :] .-
            ocean_phase_deg
        )
        for i in 1:N_CASES
    ],
    rmse_delta_phi_vs_ocean_deg = [
        sqrt(mean((
            phase_unwrapped_aligned_deg_matrix[i, :] .-
            ocean_phase_deg
        ).^2))
        for i in 1:N_CASES
    ],
    total_runtime_s = [
        sum(runtime_matrix_s[i, :])
        for i in 1:N_CASES
    ],
)

consolidated_file = joinpath(
    OUTPUT_DIR,
    "reference-efield-firi_all_cases_10wg_$(date_tag).jld2",
)

JLD2.jldsave(
    consolidated_file;
    scenario_summary = scenario_summary,
    df_ground_catalog = df_ground_catalog,
    df_cases = df_cases,
    df_phase_branch_audit = df_phase_branch_audit,
    df_summary = df_summary,
    case_keys = case_keys,
    case_labels = case_labels,
    case_types = case_types,
    case_ground_matrix = case_ground_matrix,
    amplitude_matrix = amplitude_matrix,
    phase_wrapped_raw_matrix = phase_wrapped_raw_matrix,
    phase_reference_rad_matrix = phase_reference_rad_matrix,
    phase_unwrapped_raw_matrix = phase_unwrapped_raw_matrix,
    phase_unwrapped_aligned_matrix = phase_unwrapped_aligned_matrix,
    phase_unwrapped_aligned_deg_matrix = phase_unwrapped_aligned_deg_matrix,
    branch_offset_cycles = branch_offset_cycles,
    zdt_eval = zdt_eval,
    lats_mid = lats_mid,
    lons_mid = lons_mid,
    dists_mid_m = dists_mid,
    dists_start_m = dists_start,
    dists_end_m = dists_end,
    segment_lengths_km = segment_lengths_km,
    altitude_m = collect(alt),
    nefiri_mid = nefiri_mid,
    runtime_matrix_s = runtime_matrix_s,
    worker_matrix = worker_matrix,
    individual_files = individual_files,
)

if WRITE_LEGACY_OCEAN_ALIAS
    legacy_file = "./reference-efield-firi.jld2"
    ocean_file = individual_files[ocean_index]
    cp(ocean_file, legacy_file; force=true)
    @warn "Se sobrescribió el alias histórico" legacy_file ocean_file
end

display(df_summary)
println("Archivo consolidado: ", consolidated_file)
println("Referencias individuales guardadas: ", length(individual_files))

## Uso posterior en la inversión

Cada archivo individual contiene las claves históricas:

- `amp`
- `phi`
- `zdt`

La clave `phi` **no es simplemente la fase raw devuelta por LMP**. Se le ha aplicado un único desplazamiento entero de \(2\pi\), fijado a las 00:00 UTC respecto del caso oceánico `ground_10`. De ese modo, el `DSP.unwrap` usado por el script de inversión conserva una rama comparable entre materiales sin reajustarla independientemente en cada tiempo.

Para auditar el procedimiento también se guardan:

- `phi_wrapped_raw_rad`
- `phi_unwrapped_raw_rad`
- `phi_unwrapped_aligned_rad`
- `phase_branch_offset_cycles`
- materiales y propiedades eléctricas por segmento.